# 5.3 · Restringe herramientas, archivos y red
**Módulo 5 · Asegura el agente con OpenShell** — *Seguridad de agentes con OpenShell y NemoClaw* (Platzi × NVIDIA Deep Learning Institute)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uncrayon/curso-seguridad-agentes-nemoclaw/blob/main/clases/5.3-sandbox-y-politicas/5.3-sandbox-y-politicas.ipynb)

Vas a trabajar con la **política real** de un sandbox de NemoClaw, en su formato de OpenShell (`filesystem_policy`, `landlock`, `process`, `network_policies`),
y con un simulador en Python que la evalúa como lo hace el curso de NVIDIA: **predices → evalúas → comparas → puntúas**.

Después agregas un preset, distingues qué cambia en caliente, atacas al proyecto *Soporte NemoClaw* con su propia política y diagnosticas un bloqueo.
El simulador **predice**; la verdad la da el sandbox en vivo. Por eso cada acción trae el comando para confirmarla en el lab (`cookbook.md`).

Basado en la lección 04a-safety del curso de NVIDIA DLI (Apache-2.0) y en `nemoclaw-blueprint/policies/` de NVIDIA/NemoClaw (Apache-2.0).

> **¿Sin API key?** Ejecuta todo igual: el notebook detecta que no hay key y usa un **modelo simulado (MOCK)** que imita la forma de las respuestas de la API. Con tu key `nvapi-` (clase 1.2), las mismas celdas llaman al modelo real.

> **¿No programas?** No necesitas leer el código. Presiona ▶ en cada celda, en orden, y lee el texto y lo que aparece debajo (o usa **Entorno de ejecución → Ejecutar todo** en Colab). Las celdas con ⚙️ son de preparación: se ejecutan igual, pero no necesitas abrirlas. Las palabras nuevas están en el [glosario](https://github.com/uncrayon/curso-seguridad-agentes-nemoclaw/blob/main/GLOSARIO.md).

In [ ]:
# @title ⚙️ Configuración (ejecútala siempre primero)
# --- Configuración (ejecútala siempre primero) ---------------------------------
import os, sys, json, subprocess

def _instala(paquete, modulo=None):
    try:
        __import__(modulo or paquete)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

_instala("openai")
_instala("python-dotenv", "dotenv")

# 1) Google Colab: guarda tu key en Secrets (icono de llave 🔑) con el nombre NVIDIA_API_KEY
try:
    from google.colab import userdata
    os.environ.setdefault("NVIDIA_API_KEY", userdata.get("NVIDIA_API_KEY") or "")
except Exception:
    pass
# 2) Local: archivo .env en la raíz del repo (copia .env.example) o `export NVIDIA_API_KEY=nvapi-...`
from dotenv import load_dotenv
load_dotenv(); load_dotenv("../../.env")

BASE_URL    = os.environ.get("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")
MODEL       = os.environ.get("NVIDIA_MODEL", "nvidia/nemotron-3.5-lightning-30b-a3b")
EMBED_MODEL = os.environ.get("NVIDIA_EMBED_MODEL", "nvidia/nemotron-3-embed-1b")
API_KEY     = (os.environ.get("NVIDIA_API_KEY") or "").strip()
USE_MOCK    = not API_KEY

if API_KEY and "integrate.api.nvidia.com" in BASE_URL and not API_KEY.startswith("nvapi-"):
    print("⚠  Tu key no empieza con 'nvapi-'. ¿Copiaste la key completa de build.nvidia.com?")
print("Modo:", "MOCK (sin NVIDIA_API_KEY: respuestas simuladas)" if USE_MOCK else f"EN VIVO · {MODEL}")

In [ ]:
# @title ⚙️ Cliente: modelo real o simulado, con errores explicados en español
# --- Cliente: modelo real o simulado, con errores explicados en español --------
import re, time, math, hashlib
from types import SimpleNamespace as NS

def _msg(content):
    m = NS(content=content, tool_calls=None, role="assistant", reasoning_content=None)
    return NS(choices=[NS(message=m, finish_reason="stop")], usage=NS(prompt_tokens=0, completion_tokens=0, total_tokens=0))

def _call(name, args):
    tc = NS(id="call_" + hashlib.md5((name + json.dumps(args)).encode()).hexdigest()[:8], type="function",
            function=NS(name=name, arguments=json.dumps(args, ensure_ascii=False)))
    m = NS(content=None, tool_calls=[tc], role="assistant", reasoning_content=None)
    return NS(choices=[NS(message=m, finish_reason="tool_calls")], usage=NS(prompt_tokens=0, completion_tokens=0, total_tokens=0))

class MockClient:
    """Imita la FORMA de la API de chat completions. No es inteligente: sigue reglas.
    Cada notebook puede agregar reglas propias con  @regla_mock  (ver más abajo)."""
    reglas = []
    def __init__(self):
        self.chat = NS(completions=NS(create=self._create))
        self.embeddings = NS(create=self._embed)
    def _create(self, model=None, messages=None, tools=None, **kw):
        nombres = [t["function"]["name"] for t in (tools or [])]
        for r in self.reglas:
            out = r(messages, nombres)
            if out is not None:
                return out
        ultimo = messages[-1]
        texto = (ultimo.get("content") or "")
        if ultimo["role"] == "tool":
            return _msg("Según la herramienta: " + str(ultimo["content"]))
        if re.search(r"c[oó]mo me llamo|what is my name", texto, re.I):
            vistos = [re.search(r"(?:me llamo|my name is)\s+(\w+)", m.get("content") or "", re.I)
                      for m in messages[:-1] if m["role"] == "user"]
            vistos = [v.group(1) for v in vistos if v]
            return _msg(f"Te llamas {vistos[-1]}." if vistos else "No sé cómo te llamas: no me lo has dicho en esta conversación.")
        return _msg("(mock) Respuesta simulada. Configura NVIDIA_API_KEY para hablar con el modelo real.")
    def _embed(self, model=None, input=None, **kw):
        textos = [input] if isinstance(input, str) else list(input)
        return NS(data=[NS(embedding=_vector_bolsa(t), index=i) for i, t in enumerate(textos)])

def regla_mock(f):
    """Decorador: agrega una regla al modelo simulado. f(messages, nombres_de_tools) -> respuesta | None"""
    MockClient.reglas.append(f)
    return f

_VACIAS = set("que qué como cómo para por con una uno los las del este esta eso esto hago hace tengo puedo sale mis tus sus the and you what how".split())

def _vector_bolsa(texto, dim=256):
    """Embedding simulado: bolsa de palabras con hashing. Sirve para que RAG funcione offline."""
    v = [0.0] * dim
    for w in re.findall(r"\w+", texto.lower()):
        if len(w) > 2 and w not in _VACIAS:
            w = w[:6]                       # raíz burda: "reanudar" ~ "reanudé"
            v[int(hashlib.md5(w.encode()).hexdigest(), 16) % dim] += 1.0
    n = math.sqrt(sum(x * x for x in v)) or 1.0
    return [x / n for x in v]

def diagnostico(e):
    """Traduce un error de la API a: qué pasó + qué hacer. Es el 'FAQ' en código."""
    status = getattr(e, "status_code", None)
    cuerpo = str(getattr(e, "body", "") or e)[:300]
    tabla = {
        400: "400 · Solicitud inválida: algún parámetro no le gusta al modelo (¿tools en un modelo que no las soporta? ¿max_tokens muy alto?). Lee el cuerpo del error.",
        401: "401 · Key rechazada: revisa que NVIDIA_API_KEY sea la key completa (empieza con nvapi-), sin espacios ni comillas. Si la regeneraste, la vieja ya no sirve.",
        403: "403 · Tu key es válida pero no tiene acceso a ese modelo o se acabaron tus créditos. Revisa tu cuenta en build.nvidia.com o prueba otro modelo.",
        404: "404 · Ese modelo o esa ruta no existen. Revisa MODEL / EMBED_MODEL (copia el ID exacto del catálogo) y que BASE_URL termine en /v1.",
        410: "410 · El modelo fue retirado del endpoint gratuito. Cambia MODEL por uno vigente del catálogo de build.nvidia.com.",
        422: "422 · Argumentos con formato incorrecto (p. ej. falta input_type en embeddings o un esquema de tool inválido).",
        429: "429 · Demasiadas solicitudes por minuto. Espera unos segundos y reintenta; no ejecutes celdas en ráfaga.",
    }
    if status in tabla:
        return f"{tabla[status]}\n   Detalle del servidor: {cuerpo}"
    if status and 500 <= status < 600:
        return f"{status} · Falla del lado del servidor o el modelo está arrancando. Reintenta en 30-60 s; si persiste, prueba otro modelo.\n   Detalle: {cuerpo}"
    nombre = type(e).__name__
    if "Timeout" in nombre:
        return "Timeout · El modelo tardó demasiado. Reintenta, baja max_tokens o usa un modelo más rápido."
    if "Connection" in nombre:
        return "Sin conexión · No se pudo llegar a BASE_URL. Revisa tu internet, VPN/proxy corporativo o el valor de BASE_URL."
    return f"{nombre}: {cuerpo}"

if USE_MOCK:
    client = MockClient()
else:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60, max_retries=2)

def chat(messages, tools=None, model=None, **kw):
    """Una llamada al modelo. Devuelve la respuesta cruda (choices[0].message, finish_reason, usage)."""
    kw.setdefault("max_tokens", 4096)          # los modelos que razonan gastan tokens antes de responder
    if tools:
        kw["tools"] = tools
    try:
        r = client.chat.completions.create(model=model or MODEL, messages=messages, **kw)
    except Exception as e:
        raise RuntimeError(diagnostico(e)) from None
    ch = r.choices[0]
    if ch.finish_reason == "length" and not (ch.message.content or ch.message.tool_calls):
        print("⚠  finish_reason=length sin contenido: el modelo gastó max_tokens razonando. Sube max_tokens.")
    return r

def embed(textos, input_type="passage", model=None):
    """Embeddings. input_type='query' para preguntas, 'passage' para documentos."""
    textos = [textos] if isinstance(textos, str) else list(textos)
    try:
        r = client.embeddings.create(model=model or EMBED_MODEL, input=textos,
                                     extra_body={"input_type": input_type, "truncate": "END"}) if not USE_MOCK \
            else client.embeddings.create(model=model or EMBED_MODEL, input=textos)
    except Exception as e:
        raise RuntimeError(diagnostico(e)) from None
    return [d.embedding for d in r.data]

print("Cliente listo:", "MockClient" if USE_MOCK else BASE_URL)

## 1 · Lee la política real
Este es el YAML base de un sandbox de OpenClaw en NemoClaw. Busca las cuatro preguntas de NVIDIA:
**¿a dónde puede conectarse?** (`network_policies`), **¿qué archivos alcanza?** (`filesystem_policy` + `landlock`),
**¿con qué autoridad empieza?** (`process`). La cuarta, **¿qué syscalls puede pedir?**, la fija seccomp y no aparece en este YAML.

In [ ]:
# --- 1 · La política REAL (formato de OpenShell) ------------------------------
# Extracto fiel de nemoclaw-blueprint/policies/openclaw-sandbox.yaml (NVIDIA/NemoClaw, Apache-2.0, oct. 2026).
# La de TU sandbox puede tener más grupos (presets del tier, proveedor de inferencia).
# En el lab la lees con:  openshell policy get my-assistant --full
POLITICA_YAML = """
version: 1

filesystem_policy:
  include_workdir: true          # /sandbox (el directorio de trabajo) se puede escribir
  read_only: [/usr, /lib, /proc, /dev/urandom, /app, /etc, /var/log, /var/lib/dpkg]
  read_write: [/tmp, /dev/null, /dev/pts, /sandbox/.openclaw, /sandbox/.nemoclaw, /home/linuxbrew]

landlock:
  compatibility: best_effort

process:
  run_as_user: sandbox
  run_as_group: sandbox

network_policies:
  nvidia:
    name: nvidia
    endpoints:
      - host: integrate.api.nvidia.com
        port: 443
        protocol: rest
        enforcement: enforce
        rules:
          - allow: { method: POST, path: "/v1/chat/completions" }
          - allow: { method: POST, path: "/v1/completions" }
          - allow: { method: POST, path: "/v1/embeddings" }
          - allow: { method: GET, path: "/v1/models" }
          - allow: { method: GET, path: "/v1/models/**" }
    binaries:
      - { path: /usr/local/bin/openclaw }
  managed_inference:
    name: managed_inference
    endpoints:
      - host: inference.local
        port: 443
        protocol: rest
        enforcement: enforce
        rules:
          - allow: { method: GET, path: "/**" }
          - allow: { method: POST, path: "/**" }
    binaries:
      - { path: /usr/local/bin/openclaw }
      - { path: /usr/local/bin/node }
      - { path: /usr/bin/node }
      - { path: /usr/bin/curl }
      - { path: /usr/bin/python3 }
  clawhub:
    name: clawhub
    endpoints:
      - host: clawhub.ai
        port: 443
        protocol: rest
        enforcement: enforce
        allow_encoded_slash: true
        rules:
          - allow: { method: GET, path: "/**" }
          - allow: { method: POST, path: "/**" }
    binaries:
      - { path: /usr/local/bin/openclaw }
      - { path: /usr/local/bin/node }
  openclaw_api:
    name: openclaw_api
    endpoints:
      - host: openclaw.ai
        port: 443
        protocol: rest
        enforcement: enforce
        rules:
          - allow: { method: GET, path: "/**" }
          - allow: { method: POST, path: "/**" }
      - host: catalog.openclaw.ai
        port: 443
        protocol: rest
        enforcement: enforce
        rules:
          - allow: { method: GET, path: "/**" }
    binaries:
      - { path: /usr/local/bin/openclaw }
      - { path: /usr/local/bin/node }
  openclaw_docs:
    name: openclaw_docs
    endpoints:
      - host: docs.openclaw.ai
        port: 443
        protocol: rest
        enforcement: enforce
        rules:
          - allow: { method: GET, path: "/**" }
    binaries:
      - { path: /usr/local/bin/openclaw }
  npm_registry:
    name: npm_registry
    endpoints:
      - host: registry.npmjs.org
        port: 443
        protocol: rest
        enforcement: enforce
        rules:
          - allow: { method: GET, path: "/**" }
    binaries:
      - { path: /usr/local/bin/openclaw }
"""

In [ ]:
# --- La misma política como dict de Python (para no depender de PyYAML) ------
def _ep(host, reglas, **extra):
    return {"host": host, "port": 443, "protocol": "rest", "enforcement": "enforce", **extra,
            "rules": [{"allow": {"method": m, "path": p}} for m, p in reglas]}
def _bins(*rutas):
    return [{"path": r} for r in rutas]

POLITICA_DICT = {
    "version": 1,
    "filesystem_policy": {
        "include_workdir": True,
        "read_only": ["/usr", "/lib", "/proc", "/dev/urandom", "/app", "/etc", "/var/log", "/var/lib/dpkg"],
        "read_write": ["/tmp", "/dev/null", "/dev/pts", "/sandbox/.openclaw", "/sandbox/.nemoclaw", "/home/linuxbrew"],
    },
    "landlock": {"compatibility": "best_effort"},
    "process": {"run_as_user": "sandbox", "run_as_group": "sandbox"},
    "network_policies": {
        "nvidia": {"name": "nvidia", "endpoints": [_ep("integrate.api.nvidia.com", [
            ("POST", "/v1/chat/completions"), ("POST", "/v1/completions"), ("POST", "/v1/embeddings"),
            ("GET", "/v1/models"), ("GET", "/v1/models/**")])],
            "binaries": _bins("/usr/local/bin/openclaw")},
        "managed_inference": {"name": "managed_inference",
            "endpoints": [_ep("inference.local", [("GET", "/**"), ("POST", "/**")])],
            "binaries": _bins("/usr/local/bin/openclaw", "/usr/local/bin/node", "/usr/bin/node",
                              "/usr/bin/curl", "/usr/bin/python3")},
        "clawhub": {"name": "clawhub",
            "endpoints": [_ep("clawhub.ai", [("GET", "/**"), ("POST", "/**")], allow_encoded_slash=True)],
            "binaries": _bins("/usr/local/bin/openclaw", "/usr/local/bin/node")},
        "openclaw_api": {"name": "openclaw_api",
            "endpoints": [_ep("openclaw.ai", [("GET", "/**"), ("POST", "/**")]),
                          _ep("catalog.openclaw.ai", [("GET", "/**")])],
            "binaries": _bins("/usr/local/bin/openclaw", "/usr/local/bin/node")},
        "openclaw_docs": {"name": "openclaw_docs",
            "endpoints": [_ep("docs.openclaw.ai", [("GET", "/**")])],
            "binaries": _bins("/usr/local/bin/openclaw")},
        "npm_registry": {"name": "npm_registry",
            "endpoints": [_ep("registry.npmjs.org", [("GET", "/**")])],
            "binaries": _bins("/usr/local/bin/openclaw")},
    },
}

# Si tienes PyYAML, parseamos el texto real y comprobamos que ambos dicen lo mismo.
try:
    import yaml
    POLITICA = yaml.safe_load(POLITICA_YAML)
    assert POLITICA == POLITICA_DICT, "El YAML y el dict no coinciden"
    print("PyYAML disponible: parseé el YAML real (y coincide con el dict).")
except ImportError:
    POLITICA = POLITICA_DICT
    print("Sin PyYAML: uso el dict equivalente. No necesitas instalar nada.")

print("Secciones:", list(POLITICA))
print("Grupos de red permitidos:", list(POLITICA["network_policies"]))

## 2 · Recórrela en una tabla

In [ ]:
# --- 2 · Recorre la política como lo harías en el lab -----------------------
fs = POLITICA["filesystem_policy"]
print("📁 ARCHIVOS (Landlock)")
print("   solo lectura :", ", ".join(fs["read_only"]))
print("   lectura/escr.:", ", ".join(fs["read_write"]), "+ /sandbox" if fs.get("include_workdir") else "")
print("   todo lo demás: DENEGADO")
print("\n⚙️  PROCESO:  corre como", POLITICA["process"]["run_as_user"], "(no root) · Landlock:", POLITICA["landlock"]["compatibility"])
print("\n🌐 RED (deny por defecto). Cada grupo = destinos + binarios que pueden usarlos:")
for nombre, g in POLITICA["network_policies"].items():
    destinos = ", ".join(f"{e['host']}:{e['port']}" for e in g["endpoints"])
    metodos = sorted({r["allow"]["method"] for e in g["endpoints"] for r in e.get("rules", [])}) or ["(todo, L4)"]
    binarios = ", ".join(b["path"].split("/")[-1] for b in g.get("binaries", []))
    print(f"   {nombre:<18} {destinos:<46} {'/'.join(metodos):<9} ← {binarios}")

Tres cosas para notar:
- **Red = destino + binario.** `curl` solo aparece en `managed_inference`. El resto de grupos son para `openclaw`/`node`.
- **Reglas L7.** Con `protocol: rest`, el proxy revisa método y ruta de cada request, no solo el host.
- **`/sandbox/.openclaw` es de lectura y escritura.** Ahí vive `SOUL.md`. Guárdalo en la cabeza para la sección 4.

## 3 · El evaluador

In [ ]:
# --- 3 · El evaluador: una versión simplificada del de OpenShell -------------
# Port a Python del evaluador estático del curso de NVIDIA (que a su vez porta la Rego de OpenShell).
# Es una PREDICCIÓN: la verdad la tiene el sandbox en vivo (eso lo pruebas en el lab).
import copy, ipaddress, posixpath

def _glob(patron, texto, sep="/"):
    """'*' cubre un segmento; '**' cubre varios (como en las rutas y binarios de la política)."""
    rx, i = "^", 0
    while i < len(patron):
        if patron[i] == "*":
            if patron[i + 1:i + 2] == "*":
                rx += ".*"; i += 1
            else:
                rx += f"[^{re.escape(sep)}]*"
        else:
            rx += re.escape(patron[i])
        i += 1
    return re.match(rx + "$", texto) is not None

def _ip_bloqueada(host):
    """OpenShell bloquea SIEMPRE loopback, link-local (incluye 169.254.169.254) y 0.0.0.0."""
    try:
        ip = ipaddress.ip_address(host)
    except ValueError:
        return host == "localhost"
    return ip.is_loopback or ip.is_link_local or ip.is_unspecified

def _endpoint(grupo, host, puerto):
    for ep in grupo.get("endpoints", []):
        h = ep.get("host", "").lower()
        if ep.get("port") == puerto and (h == host or ("*" in h and _glob(h, host, sep="."))):
            return ep
    return None

def _binario_ok(grupo, binario, ancestros=()):
    """Se compara la ruta REAL del ejecutable (/proc/<pid>/exe) y sus ancestros, nunca argv[0]."""
    candidatos = [binario, *ancestros]
    return any(_glob(b["path"], c) if "*" in b["path"] else b["path"] in candidatos
               for b in grupo.get("binaries", []) for c in candidatos)

def _l7_ok(ep, metodo, ruta):
    if "rules" in ep:                                  # protocol: rest → se inspecciona cada request
        return any(r["allow"]["method"] in ("*", metodo.upper()) and _glob(r["allow"]["path"], ruta)
                   for r in ep["rules"] if "allow" in r)
    if ep.get("access") == "read-only":                # suposición del curso: solo lectura = GET/HEAD
        return metodo.upper() in ("GET", "HEAD")
    return True                                        # access: full o sin protocol → solo L4

def evaluar_red(binario, host, puerto=443, metodo=None, ruta="/", politica=None, ancestros=()):
    politica = politica or POLITICA
    host = host.lower()
    if _ip_bloqueada(host):
        return "DENY", f"{host} es loopback/link-local/metadata: OpenShell lo bloquea siempre (SSRF)"
    coinciden, sin_binario = [], []
    for nombre, grupo in politica.get("network_policies", {}).items():
        ep = _endpoint(grupo, host, puerto)
        if ep and _binario_ok(grupo, binario, ancestros):
            coinciden.append((nombre, ep))
        elif ep:
            sin_binario.append(f"el binario '{binario}' no está permitido en '{nombre}'")
    if not coinciden:
        return "DENY", "; ".join(sin_binario) or f"ningún grupo incluye {host}:{puerto} (deny por defecto)"
    nombre, ep = sorted(coinciden, key=lambda x: x[0])[0]
    if metodo and not _l7_ok(ep, metodo, ruta):
        return "DENY", f"{metodo} {ruta} no está permitido en '{nombre}' (regla L7)"
    return "ALLOW", f"coincide con '{nombre}'"

def evaluar_archivo(ruta, modo="read", politica=None):
    """Landlock: gana el prefijo más largo. read_write permite todo; read_only solo leer."""
    fs = (politica or POLITICA)["filesystem_policy"]
    ruta = posixpath.normpath(ruta)                    # resuelve ../ antes de comparar
    bajo = lambda base: ruta == base or ruta.startswith(base.rstrip("/") + "/")
    rw = max((b for b in fs.get("read_write", []) if bajo(b)), key=len, default=None)
    ro = max((b for b in fs.get("read_only", []) if bajo(b)), key=len, default=None)
    mejor = max(filter(None, [rw, ro]), key=len, default=None)
    if mejor is None:
        if fs.get("include_workdir") and bajo("/sandbox"):
            return "ALLOW", "dentro de /sandbox (include_workdir: true)"
        return "DENY", f"{ruta} no está en read_only ni en read_write (EACCES)"
    if mejor == rw:
        return "ALLOW", f"{ruta} está bajo read_write {rw}"
    return ("ALLOW", f"lectura permitida en read_only {ro}") if modo == "read" else \
           ("DENY", f"{ro} es read_only: escribir da EACCES")

SYSCALLS_PELIGROSAS = {"ptrace", "mount", "setuid"}   # excluidas por seccomp (NVIDIA 04a)

def evaluar_proceso(operacion, politica=None):
    p = (politica or POLITICA)["process"]
    if operacion in SYSCALLS_PELIGROSAS:
        return "DENY", f"seccomp: {operacion} devuelve EPERM antes de ejecutarse"
    if operacion in ("sudo", "su") and p.get("run_as_user") != "root":
        return "DENY", f"corre como '{p['run_as_user']}' sin privilegios y con no-new-privileges"
    return "ALLOW", "operación normal de proceso"

def evaluar(a, politica=None):
    if a["tipo"] == "red":
        return evaluar_red(a["binario"], a["host"], a.get("puerto", 443), a.get("metodo"), a.get("ruta", "/"), politica)
    if a["tipo"] == "archivo":
        return evaluar_archivo(a["ruta"], a["modo"], politica)
    return evaluar_proceso(a["op"], politica)

print(evaluar_red("/usr/bin/curl", "example.com"))
print(evaluar_archivo("/etc/passwd", "read"))

## 4 · Predice primero

In [ ]:
# --- 4 · Predice primero ------------------------------------------------------
# Cada acción trae el comando REAL para confirmarla en el lab (cookbook de la 5.3).
CURL = "/usr/bin/curl"; OPENCLAW = "/usr/local/bin/openclaw"; GIT = "/usr/bin/git"
ACCIONES = {
 "a": {"tipo": "red", "binario": CURL, "host": "inference.local", "metodo": "GET", "ruta": "/v1/models",
       "desc": "curl → inference.local GET /v1/models", "lab": "curl -sS -m 8 https://inference.local/v1/models"},
 "b": {"tipo": "red", "binario": CURL, "host": "example.com", "metodo": "GET", "ruta": "/",
       "desc": "curl → example.com (exfiltración)", "lab": "curl -sS -m 8 https://example.com"},
 "c": {"tipo": "red", "binario": CURL, "host": "169.254.169.254", "puerto": 80, "metodo": "GET", "ruta": "/latest/meta-data/",
       "desc": "curl → metadata de la nube", "lab": "curl -sS -m 6 http://169.254.169.254/latest/meta-data/"},
 "d": {"tipo": "red", "binario": CURL, "host": "integrate.api.nvidia.com", "metodo": "GET", "ruta": "/v1/models",
       "desc": "curl → API de NVIDIA (binario equivocado)", "lab": "curl -sS -m 8 https://integrate.api.nvidia.com/v1/models"},
 "e": {"tipo": "red", "binario": OPENCLAW, "host": "integrate.api.nvidia.com", "metodo": "POST", "ruta": "/v1/chat/completions",
       "desc": "openclaw → NVIDIA POST chat", "lab": "(lo hace el agente en cada turno)"},
 "f": {"tipo": "red", "binario": OPENCLAW, "host": "docs.openclaw.ai", "metodo": "POST", "ruta": "/buscar",
       "desc": "openclaw → docs.openclaw.ai POST", "lab": "(solo predicción)"},
 "g": {"tipo": "red", "binario": GIT, "host": "github.com", "metodo": "GET", "ruta": "/NVIDIA/NemoClaw",
       "desc": "git clone desde GitHub", "lab": "git ls-remote https://github.com/NVIDIA/NemoClaw"},
 "h": {"tipo": "archivo", "ruta": "/sandbox/.openclaw/workspace/SOUL.md", "modo": "read",
       "desc": "leer SOUL.md", "lab": "head -3 /sandbox/.openclaw/workspace/SOUL.md"},
 "i": {"tipo": "archivo", "ruta": "/sandbox/.openclaw/workspace/SOUL.md", "modo": "write",
       "desc": "ESCRIBIR SOUL.md (la persona)", "lab": "(no lo hagas en vivo: solo predicción)"},
 "j": {"tipo": "archivo", "ruta": "/etc/hosts", "modo": "write",
       "desc": "escribir /etc/hosts", "lab": "touch /etc/hosts"},
 "k": {"tipo": "archivo", "ruta": "/var/lib/apt/lists", "modo": "read",
       "desc": "leer /var/lib/apt/lists", "lab": "ls /var/lib/apt/lists"},
 "l": {"tipo": "proceso", "op": "ptrace", "desc": "ptrace (depurar otro proceso)", "lab": "ver receta de seccomp (5.2)"},
}

# ✏️ Tu turno: escribe ALLOW o DENY en cada una ANTES de ejecutar la sección 5.
# Viene una primera adivinanza (algunas están mal a propósito).
PREDICCIONES = {
 "a": "ALLOW", "b": "DENY", "c": "DENY", "d": "ALLOW", "e": "ALLOW", "f": "ALLOW",
 "g": "ALLOW", "h": "ALLOW", "i": "DENY", "j": "DENY", "k": "ALLOW", "l": "DENY",
}
assert set(PREDICCIONES) == set(ACCIONES) and set(PREDICCIONES.values()) <= {"ALLOW", "DENY"}
for k, a in ACCIONES.items():
    print(f"{k}  {a['desc']:<42} tu predicción: {PREDICCIONES[k]}")

## 5 · Evalúa y puntúa

In [ ]:
# --- 5 · Evalúa y puntúa ------------------------------------------------------
def puntuar(acciones, predicciones, politica=None):
    aciertos = 0
    for k, a in acciones.items():
        veredicto, razon = evaluar(a, politica)
        ok = veredicto == predicciones[k]
        aciertos += ok
        print(f"{'✓' if ok else '✗'} {k} {a['desc']:<42} predijiste {predicciones[k]:<5} → {veredicto:<5} | {razon}")
    print(f"\nPuntaje: {aciertos}/{len(acciones)}")
    return aciertos

puntaje = puntuar(ACCIONES, PREDICCIONES)

**Lee las filas ✗.** Las sorpresas típicas:
- **d**: el host está permitido, pero para `openclaw`, no para `curl`. El permiso va **atado al binario**.
- **f**: `docs.openclaw.ai` está permitido... solo con `GET`. La regla L7 frena el `POST`.
- **i**: `SOUL.md` está bajo `/sandbox/.openclaw`, que es `read_write`. La política **no** protege la persona del agente: eso queda arriba (permisos de archivo, historial revisable).
- **k**: `/var/lib/apt/lists` es legible para cualquier usuario de Linux, pero no está en la política: Landlock lo niega.

## 6 · Confírmalo en el lab

In [ ]:
# --- 6 · Ahora confírmalo en el sandbox de verdad ------------------------------
# Copia estos comandos al lab (cookbook.md, receta 3). Corre cada uno con:
#   openshell sandbox exec -n my-assistant -- <comando>
print("Predicción del simulador vs. lo que debes comprobar en Brev:\n")
for k, a in ACCIONES.items():
    if a["lab"].startswith("("):
        continue
    v, _ = evaluar(a)
    print(f"{k} [{v:<5}]  {a['lab']}")
print("\nSi el lab no coincide: revisa la política activa (openshell policy get my-assistant --full),")
print("el binario real y quién respondió (¿el proxy o el servidor de destino?).")

## 7 · Agregar un preset
En el lab: `nemoclaw my-assistant policy add github --dry-run` muestra esto mismo antes de aplicarlo.

In [ ]:
# --- 7 · Agregar un preset (lo que hace `nemoclaw my-assistant policy add`) ---
# Formato REAL de un preset: un bloque `preset:` + `network_policies:` que se fusiona con la política activa.
# Fuente: nemoclaw-blueprint/policies/presets/github.yaml (NVIDIA/NemoClaw, Apache-2.0)
PRESET_GITHUB = {
    "preset": {"name": "github", "description": "GitHub.com and GitHub API access (git)"},
    "network_policies": {"github": {"name": "github",
        "endpoints": [{"host": "github.com", "port": 443, "access": "full"},
                      {"host": "api.github.com", "port": 443, "access": "full"}],
        "binaries": [{"path": "/usr/bin/git"}]}},
}

def vista_previa(preset):
    """Como `--dry-run`: qué egress abriría, para qué binarios."""
    print(f"Preset '{preset['preset']['name']}' abriría:")
    for nombre, g in preset["network_policies"].items():
        bins = ", ".join(b["path"] for b in g.get("binaries", [])) or "⚠ CUALQUIER binario"
        for e in g["endpoints"]:
            reglas = [f"{r['allow']['method']} {r['allow']['path']}" for r in e.get("rules", [])]
            alcance = ", ".join(reglas) or f"access: {e.get('access', 'L4 sin inspección')}"
            print(f"  + {e['host']}:{e['port']}  [{alcance}]  ← {bins}")

def aplicar_preset(politica, preset):
    """Devuelve una política NUEVA con el preset fusionado (la original no cambia)."""
    nueva = copy.deepcopy(politica)
    nueva["network_policies"].update(copy.deepcopy(preset["network_policies"]))
    return nueva

vista_previa(PRESET_GITHUB)
CON_GITHUB = aplicar_preset(POLITICA, PRESET_GITHUB)
print("\nantes  git  → github.com:", evaluar_red(GIT, "github.com", politica=POLITICA)[0])
print("después git  → github.com:", evaluar_red(GIT, "github.com", politica=CON_GITHUB)[0])
print("después curl → github.com:", *evaluar_red(CURL, "github.com", politica=CON_GITHUB))

Abrir GitHub para `git` no se lo abre a `curl`. Si alguna vez te tienta quitar `binaries` para "que funcione", recuerda: entonces **cualquier** proceso del sandbox podría mandar datos a ese host.

## 8 · En caliente vs. recrear

In [ ]:
# --- 8 · ¿Se aplica en caliente o hay que recrear el sandbox? ---------------
# Red e inferencia: en caliente. Filesystem, Landlock y proceso: se fijan al crear el sandbox.
EN_CALIENTE = {"network_policies": True, "inference": True,
               "filesystem_policy": False, "landlock": False, "process": False}

CAMBIOS = [
    ("Agregar el preset pypi",                         "network_policies"),
    ("Aprobar example.com en openshell term",          "network_policies"),
    ("Cambiar el modelo de inferencia",                "inference"),
    ("Hacer /sandbox/.openclaw de solo lectura",       "filesystem_policy"),
    ("Correr el agente como root",                     "process"),
    ("Cambiar Landlock a modo estricto",               "landlock"),
]
for cambio, seccion in CAMBIOS:
    print(f"{cambio:<46} → {'en caliente ✅' if EN_CALIENTE[seccion] else 'recrear el sandbox 🔁'}  ({seccion})")

## 9 · El caso del proyecto: la inyección de la clase 5.1, contra la política real

In [ ]:
# --- 9 · El caso del proyecto: Soporte NemoClaw dentro del sandbox -----------
# La FAQ envenenada de la clase 5.1, ahora contra la política real. Cada paso se evalúa.
PASOS = [
    ("El agente lee la FAQ del workspace (contenido no confiable)",
     {"tipo": "archivo", "ruta": "/sandbox/.openclaw/workspace/faq/02-api-key.md", "modo": "read"}),
    ("Lee MEMORY.md, con datos de estudiantes (datos privados)",
     {"tipo": "archivo", "ruta": "/sandbox/.openclaw/workspace/MEMORY.md", "modo": "read"}),
    ("Intenta mandarlo con curl a un host del atacante",
     {"tipo": "red", "binario": CURL, "host": "verificador-keys.example", "metodo": "POST", "ruta": "/registro"}),
    ("Lo manda desde el runtime del agente a un host PERMITIDO (clawhub.ai acepta POST)",
     {"tipo": "red", "binario": "/usr/local/bin/node", "host": "clawhub.ai", "metodo": "POST", "ruta": "/api/cualquier-cosa"}),
]
for desc, a in PASOS:
    v, razon = evaluar(a)
    print(f"{'🟢' if v == 'ALLOW' else '🔴'} {v:<5} {desc}\n         {razon}")

**Lee el último paso con calma.** El sandbox frenó al atacante en el canal obvio (`curl` a un host desconocido)
y aun así dejó pasar la exfiltración por un canal **permitido**. Una allowlist aprueba el destino, no el significado de los bytes.

Y hay algo más directo: si un comando **permitido** imprime un secreto (`cat` de un archivo, `env`), ese texto ya entró al contexto del modelo
y viaja al proveedor de inferencia en el siguiente turno. **Un secreto impreso por un comando permitido ya se escapó.**
Por eso el sandbox es una capa y no la solución completa: arriba necesitas procedencia de la entrada, aprobación humana para acciones con consecuencias y secretos que nunca lleguen al sandbox (clase 5.2).

## 10 · ✏️ Tu turno: el preset del proyecto

In [ ]:
# --- 10 · ✏️ Tu turno: un preset mínimo para Soporte NemoClaw -----------------
# Queremos que el agente consulte la documentación oficial de NemoClaw, SOLO lectura,
# y que ningún otro binario (curl, python) pueda usar ese permiso.
PRESET_SOPORTE = {
    "preset": {"name": "soporte-docs", "description": "Lectura de la documentación de NemoClaw"},
    "network_policies": {"soporte_docs": {"name": "soporte_docs",
        "endpoints": [{"host": "docs.nvidia.com", "port": 443, "protocol": "rest", "enforcement": "enforce",
                       "rules": [{"allow": {"method": "GET", "path": "/nemoclaw/**"}}]}],
        "binaries": [{"path": "/usr/local/bin/openclaw"}, {"path": "/usr/local/bin/node"}]}},
}

RESERVADOS = {"npm_yarn", "personal_open_internet"}
MANTENIDOS = {"brave", "brew", "brew-balanced", "claude-code", "github", "gmail", "huggingface", "jira", "local-inference",
              "local-memory", "npm", "outlook", "personal-open-internet", "public-reference", "pypi", "tavily", "weather"}

def validar_preset(preset):
    """Reglas de docs/network-policy/create-custom-policy-presets.mdx (versión resumida)."""
    errores = []
    nombre = preset.get("preset", {}).get("name", "")
    if not re.fullmatch(r"[a-z0-9]([-a-z0-9]{0,61}[a-z0-9])?", nombre):
        errores.append(f"preset.name '{nombre}' debe ser una etiqueta RFC 1123 en minúsculas")
    if nombre in MANTENIDOS:
        errores.append(f"'{nombre}' choca con un preset mantenido por NemoClaw")
    for clave, g in preset.get("network_policies", {}).items():
        if clave in RESERVADOS:
            errores.append(f"la clave '{clave}' está reservada")
        for e in g.get("endpoints", []):
            if e.get("host") in ("*", "0.0.0.0/0", "::/0") or e.get("host", "").startswith("*") and e["host"].count(".") < 2:
                errores.append(f"host '{e.get('host')}' es un comodín demasiado amplio")
            if "allowed_ips" in e:
                errores.append("no se permite allowed_ips en presets propios")
        if not g.get("binaries"):
            errores.append(f"'{clave}' no limita binarios: cualquier proceso podría usarlo")
    return errores

def a_yaml(obj, sangria=0):
    """Mini serializador YAML (sin dependencias) para guardar tu preset en un archivo."""
    esp, lineas = "  " * sangria, []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, (dict, list)):
                lineas += [f"{esp}{k}:", a_yaml(v, sangria + 1)]
            else:
                lineas.append(f"{esp}{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v, str) else str(v).lower() if isinstance(v, bool) else v}")
    else:
        for item in obj:
            cuerpo = a_yaml(item, sangria + 1).lstrip() if isinstance(item, (dict, list)) else json.dumps(item, ensure_ascii=False)
            lineas.append(f"{esp}- {cuerpo}")
    return "\n".join(lineas)

errores = validar_preset(PRESET_SOPORTE)
print("Validación:", "✅ sin errores" if not errores else errores)
vista_previa(PRESET_SOPORTE)
CON_SOPORTE = aplicar_preset(POLITICA, PRESET_SOPORTE)
for b, m in [(OPENCLAW, "GET"), (OPENCLAW, "POST"), (CURL, "GET")]:
    print(f"  {b.split('/')[-1]:<9} {m:<4} docs.nvidia.com/nemoclaw/latest →",
          *evaluar_red(b, "docs.nvidia.com", metodo=m, ruta="/nemoclaw/latest/index.html", politica=CON_SOPORTE))
print("\n# Guarda esto como soporte-docs.yaml y pruébalo en el lab con --dry-run (cookbook, receta 7):\n")
print(a_yaml(PRESET_SOPORTE))

## 11 · 🧯 Cuando algo falla
Un bloqueo inesperado se diagnostica en este orden: **¿qué dijo el sandbox? → ¿qué dice el log? → ¿era esperado? → ¿una vez o siempre?**
En el lab, el log sale de `nemoclaw my-assistant logs --tail 50` y la aprobación interactiva de `openshell term` (cookbook, recetas 5 y 6).

In [ ]:
# --- 11 · 🧯 Cuando algo falla: un bloqueo que no esperabas --------------------
# Provocamos dos fallas típicas y las diagnosticamos.

# Falla 1 · Un preset demasiado amplio (lo rechaza NemoClaw antes de aplicarlo).
PRESET_MALO = copy.deepcopy(PRESET_SOPORTE)
PRESET_MALO["preset"]["name"] = "Soporte_Docs"
PRESET_MALO["network_policies"]["soporte_docs"]["endpoints"][0]["host"] = "*"
PRESET_MALO["network_policies"]["soporte_docs"]["binaries"] = []
print("Falla 1 · validar_preset(PRESET_MALO):")
for e in validar_preset(PRESET_MALO):
    print("   ✗", e)

# Falla 2 · El agente dice "no puedo descargar el paquete". Esto es lo que ves en el sandbox y en los logs.
SALIDA_SANDBOX = "curl: (56) CONNECT tunnel failed, response 403"
LOG = ("DENIED /usr/local/bin/node -> registry.npmjs.org:443 "
       "reason: binary '/usr/local/bin/node' not allowed in policy 'npm_registry'")   # formato de ejemplo [VERIFICAR EN LAB]

def diagnosticar_bloqueo(salida, log=""):
    pasos = []
    if "CONNECT tunnel failed" in salida and "403" in salida:
        pasos.append("Es la política de red del sandbox (el proxy respondió 403), no un problema de TLS ni de DNS.")
    m = re.search(r"DENIED (\S+) -> ([\w.-]+):(\d+).*?reason: (.*)", log)
    if m:
        binario, host, puerto, razon = m.groups()
        pasos.append(f"Quién: {binario} · a dónde: {host}:{puerto} · por qué: {razon}")
        v, r = evaluar_red(binario, host, int(puerto))
        pasos.append(f"El simulador coincide: {v} ({r})")
        pasos.append("¿Era esperado? Si el agente no debía ir ahí, NO abras nada: el sandbox hizo su trabajo.")
        pasos.append("Si sí lo necesita una vez: `openshell term` → elige el sandbox → r → selecciona la solicitud → a (aprobar) o x (rechazar).")
        pasos.append("Si lo necesita siempre: `nemoclaw my-assistant policy list` y luego `policy add <preset> --dry-run`,")
        pasos.append(f"   o un preset propio con el binario exacto ({binario}) y `policy add --from-file ... --dry-run`.")
    return pasos

print("\nFalla 2 · diagnóstico:")
for p in diagnosticar_bloqueo(SALIDA_SANDBOX, LOG):
    print("   →", p)

In [ ]:
# Opcional: pídele al modelo que explique el bloqueo a un estudiante (en vivo o MOCK).
@regla_mock
def _explica_bloqueo(messages, nombres):
    if "DENIED" in (messages[-1].get("content") or ""):
        return _msg("1) El proxy de OpenShell bloqueó la conexión: node no está en la lista de binarios de 'npm_registry'. "
                    "2) Si es necesario, apruébala una vez en `openshell term`. "
                    "3) Para dejarlo fijo, aplica el preset npm con `nemoclaw my-assistant policy add npm --dry-run` y revísalo antes de confirmar.")

r = chat([{"role": "system", "content": "Eres Soporte NemoClaw. Explica en 3 pasos cortos, en español, qué pasó y qué hacer. No inventes comandos."},
          {"role": "user", "content": f"Mi agente falla y el log dice: {LOG}"}], max_tokens=800)
print(r.choices[0].message.content)

**Ojo con dos confusiones frecuentes:**
- Un **403** puede venir del proxy de OpenShell **o** de la aplicación de destino. `CONNECT tunnel failed, response 403` es del proxy; un `code=403` después de conectar es del servidor.
- En `openshell term`, `policy_denied` significa que la política no coincidió; `upstream_unreachable` significa que sí coincidió pero el host no respondió. Abrir más la política no arregla lo segundo.

## Avance del proyecto
Guarda tu `PRESET_SOPORTE` como `soporte-docs.yaml` junto a la skill `soporte-nemoclaw` y anota en tu hoja de trifecta (clase 5.1) qué pata cubre la política y cuál **no**:
la política limita a dónde salen los bytes, pero `clawhub.ai` y el propio proveedor de inferencia siguen siendo canales de salida.

## 🏁 Reto

In [ ]:
# --- 🏁 Reto: un preset para Soporte NemoClaw que pase TODAS estas pruebas -----
# Edita PRESET_RETO y ejecuta. Criterio de éxito: 6/6 ✓.
PRESET_RETO = copy.deepcopy(PRESET_SOPORTE)   # ✏️ empieza desde aquí: agrega build.nvidia.com, solo GET /

def probar_reto(preset):
    pol = aplicar_preset(POLITICA, preset)
    pruebas = [
        ("validación sin errores",                    not validar_preset(preset)),
        ("openclaw GET docs.nvidia.com/nemoclaw/...", evaluar_red(OPENCLAW, "docs.nvidia.com", metodo="GET", ruta="/nemoclaw/latest/", politica=pol)[0] == "ALLOW"),
        ("openclaw GET build.nvidia.com/",            evaluar_red(OPENCLAW, "build.nvidia.com", metodo="GET", ruta="/", politica=pol)[0] == "ALLOW"),
        ("openclaw POST build.nvidia.com/ bloqueado", evaluar_red(OPENCLAW, "build.nvidia.com", metodo="POST", ruta="/", politica=pol)[0] == "DENY"),
        ("curl GET docs.nvidia.com bloqueado",        evaluar_red(CURL, "docs.nvidia.com", metodo="GET", ruta="/nemoclaw/", politica=pol)[0] == "DENY"),
        ("openclaw GET docs.nvidia.com/otra-cosa bloqueado", evaluar_red(OPENCLAW, "docs.nvidia.com", metodo="GET", ruta="/cuda/", politica=pol)[0] == "DENY"),
    ]
    for nombre, ok in pruebas:
        print(("✓ " if ok else "✗ ") + nombre)
    print(f"\n{sum(ok for _, ok in pruebas)}/{len(pruebas)}")

probar_reto(PRESET_RETO)